# MAGNet — Demo Inference Notebook

**Multi-person Motion Generation with DFOT + VQ-VAE**

This notebook walks through the full inference pipeline:

1. **Configuration** — Set paths, sampling task, and parameters
2. **Model Loading** — Load the DFOT network and normalization stats
3. **Data Loading** — Load motion sequences from HDF5
4. **Inference** — Generate multi-person motion samples
5. **Post-processing & Saving** — Denormalize, convert transforms, save `.npz` files
6. **Viser 3D Visualization** — Launch an interactive 3D viewer in-browser
7. **Inspect Results** — Sanity-check saved `.npz` contents

### Supported Sampling Tasks

| Task | Description |
|------|-------------|
| `joint` | Joint multi-person motion generation |
| `partner_prediction` | Predict one person given the other's full GT |
| `agentic_turn_taking` | Asynchronous leader-follower generation |
| `agentic_sync` | Synchronous agentic generation |
| `motion_control_live` | Generate person B conditioned on person A's GT (past + current) |
| `partner_inpainting` | Inpaint one person given the other's full sequence |
| `inbetweening` | Fill motion between keyframes |

## 0. Imports

In [1]:
import dataclasses
import time
import sys
from typing import List
from pathlib import Path

import numpy as np
import torch
import os

from omegaconf import OmegaConf


_viz_dir = str(Path("libs/viz").resolve())
if _viz_dir not in sys.path:
    sys.path.insert(0, _viz_dir)

from libs.train.dfot_train import load_cfg
from libs.inference.dfot_inference import load_inf_cfg
from libs.dataloaders import DataType, TrainingData, load_from_hdf5, padding_training_data
from libs.model.dfot.config import DFOTSamplingConfig
from libs.model.vqvae import pose_vqvae
from libs.model.dfot import network
from libs.utils.random_seed import set_seed
from libs.utils.root_transform_processor import RootTransformProcessor
from libs.utils.transforms import SE3, SO3

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")

PyTorch version: 2.8.0+cu128
CUDA available:  True
GPU:             NVIDIA RTX 6000 Ada Generation


## 1. Configuration

You have **two options** to configure the inference run:

- **Option A:** Load from an existing YAML config file (set `CONFIG_YAML` below)
- **Option B:** Set all parameters directly in the cells below (set `CONFIG_YAML = None`)

In [2]:
# ============================================================================
# OPTION A: Load from YAML config (set to None to use Option B instead)
# ============================================================================
CONFIG_YAML = "configs/inference/dfot/dd100.yaml"

# ============================================================================
# OPTION B: Set parameters directly (only used when CONFIG_YAML = None)
# ============================================================================

# --- Paths ---
CHECKPOINT_DIR = "./checkpoints/dfot/magnet_dd100/v0"
EPOCH          = "500K"
DATASET_PATH   = "./data/duolandosmplx_dataset.hdf5"
SMPL_PATH      = "./data/smplx/SMPLX_NEUTRAL.npz"
OUTPUT_DIR     = "./outputs/demo"

# --- Data ---
DATASET_SPLIT  = "test"
DATA_LIST      = [
    "PasoDable_005_005",
    "Qiaqiaqia_005_002",
]

# --- Sampling task ---
# Options: "joint", "partner_prediction", "agentic_turn_taking",
#          "agentic_sync", "motion_control_live", "partner_inpainting",
#          "inbetweening"
SAMPLING_TASK = "joint"

# --- Sampling parameters ---
SAMPLING_STEPS    = 50    # denoising steps
SAMPLING_NUM      = 10    # number of samples to generate
CONTEXT_SEQ_LEN   = 32    # context (conditioning) frames (must be multiple of 4, min 4)
SAMPLING_SEQ_LEN  = 200   # total frames to generate
SAMPLING_SUBSEQ_LEN = 16  # sub-sequence length per denoising window
AR_SEQ_STRIDE     = 16    # autoregressive stride

# --- Denoising ---
DENOISING_PROCESS = "ddim"
DDIM_ETA          = 0.0

# --- Classifier-free guidance ---
CFG_CLEAN = 1.0

# --- Advanced ---
MASK_ADDITIONAL_PERSON    = True
PERSON_FLIP               = False
GUIDANCE_OPTIMIZATION     = False
UPDATE_HISTORY_TRANSFORMS = True
INBETWEENING_KEYFRAMES    = [0, 63]

# --- Reproducibility ---
RANDOM_SEED = 1111

### Build the configuration

In [3]:
# Task-specific presets (schedule + root transform mode)
TASK_PRESETS = {
    "joint": dict(
        sampling_task="joint",
        sampling_schedule="causal_uncertainty",
        root_transform_mode="temporal_partner",
    ),
    "partner_prediction": dict(
        sampling_task="partner_prediction",
        sampling_schedule="causal_uncertainty",
        root_transform_mode="temporal_partner",
    ),
    "agentic_turn_taking": dict(
        sampling_task="agentic_turn_taking",
        sampling_schedule="causal_uncertainty",
        root_transform_mode="temporal",
    ),
    "agentic_sync": dict(
        sampling_task="agentic_sync",
        sampling_schedule="causal_uncertainty",
        root_transform_mode="temporal",
    ),
    "motion_control_live": dict(
        sampling_task="motion_control_live",
        sampling_schedule="causal_uncertainty",
        root_transform_mode="temporal",
    ),
    "partner_inpainting": dict(
        sampling_task="partner_inpainting",
        sampling_schedule="full_sequence",
        root_transform_mode="temporal_partner",
    ),
    "inbetweening": dict(
        sampling_task="inbetweening",
        sampling_schedule="causal_uncertainty",
        root_transform_mode="temporal",
    ),
}

if CONFIG_YAML is not None:
    # ------ Option A: load from YAML ------
    cfg = load_inf_cfg(CONFIG_YAML)
    print(f"Loaded config from: {CONFIG_YAML}")
else:
    # ------ Option B: build from variables ------
    preset = TASK_PRESETS[SAMPLING_TASK]

    sampling_cfg = DFOTSamplingConfig(
        sampling_task=preset["sampling_task"],
        sampling_schedule=preset["sampling_schedule"],
        context_seq_len=CONTEXT_SEQ_LEN,
        sampling_seq_len=SAMPLING_SEQ_LEN,
        sampling_subseq_len=SAMPLING_SUBSEQ_LEN,
        ar_seq_stride=AR_SEQ_STRIDE,
        sampling_steps=SAMPLING_STEPS,
        sampling_num=SAMPLING_NUM,
        denoising_process=DENOISING_PROCESS,
        ddim_eta=DDIM_ETA,
        cfg_scale_dict={"clean": CFG_CLEAN},
        is_guidance_optimization=GUIDANCE_OPTIMIZATION,
        is_update_history_transforms=UPDATE_HISTORY_TRANSFORMS,
        root_transform_mode=preset["root_transform_mode"],
        inbetweening_key_frame_indices=INBETWEENING_KEYFRAMES,
    )

    # Resolve checkpoint directory
    checkpoint_dir = Path(CHECKPOINT_DIR)
    if EPOCH == "best":
        checkpoint_dir = checkpoint_dir / "best_checkpoints"
    else:
        epoch_str = EPOCH.replace("K", "000").replace("k", "000")
        checkpoint_dir = checkpoint_dir / f"checkpoints_{epoch_str}"
        if not checkpoint_dir.exists():
            checkpoint_dir = checkpoint_dir.parent / f"save_checkpoints_{epoch_str}"

    cfg = OmegaConf.create({
        "output_dir": OUTPUT_DIR,
        "checkpoint_dir": str(checkpoint_dir),
        "epoch": EPOCH,
        "dataset_path": DATASET_PATH,
        "smpl_path": SMPL_PATH,
        "dataset_split": DATASET_SPLIT,
        "inference_data_list": DATA_LIST,
        "is_mask_additional_person": MASK_ADDITIONAL_PERSON,
        "is_person_flip": PERSON_FLIP,
        "result_dir_prefix": f"{EPOCH}_{DATASET_SPLIT}",
        "save_motion_data": True,
        "random_seed": RANDOM_SEED,
        "sampling_cfg": OmegaConf.structured(sampling_cfg),
    })
    cfg = OmegaConf.to_object(cfg)
    cfg = OmegaConf.create(cfg)
    print("Built config from notebook variables.")

# Print summary
print("\n" + "=" * 60)
print("  Configuration Summary")
print("=" * 60)
print(f"  Checkpoint:      {cfg.checkpoint_dir}")
print(f"  Dataset:         {cfg.dataset_path}")
print(f"  Task:            {cfg.sampling_cfg.sampling_task}")
print(f"  Schedule:        {cfg.sampling_cfg.sampling_schedule}")
print(f"  Denoising:       {cfg.sampling_cfg.denoising_process} (steps={cfg.sampling_cfg.sampling_steps})")
print(f"  Samples:         {cfg.sampling_cfg.sampling_num}")
print(f"  Context frames:  {cfg.sampling_cfg.context_seq_len}")
print(f"  Generate frames: {cfg.sampling_cfg.sampling_seq_len}")
print(f"  Sequences:       {cfg.inference_data_list}")
print("=" * 60)

Loaded config from: configs/inference/dfot/dd100.yaml

  Configuration Summary
  Checkpoint:      checkpoints/dfot/magnet_dd100/v0/checkpoints_500000
  Dataset:         data/duolandosmplx_dataset.hdf5
  Task:            joint
  Schedule:        causal_uncertainty
  Denoising:       ddim (steps=50)
  Samples:         10
  Context frames:  32
  Generate frames: 200
  Sequences:       ['PasoDable_005_005', 'Qiaqiaqia_005_002']


## 2. Load Model

Load the DFOT network (with frozen VQ-VAE) and normalization statistics.

In [4]:
set_seed(cfg.random_seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_dir = Path(cfg.checkpoint_dir)
model_path  = checkpoint_dir / "model.safetensors"
config_path = checkpoint_dir / "../config.yaml"

print(f"Loading training config from: {config_path}")
config = load_cfg(str(config_path))

print(f"Loading model from: {model_path}")
model = network.DFOTNetwork.load(model_path, config.model_cfg, device)

# Load normalization statistics
ms_data = np.load(config.mean_std_path)
mean = torch.from_numpy(ms_data["mean"]).to(torch.float32).to(device)
std  = torch.from_numpy(ms_data["std"]).to(torch.float32).to(device)

print(f"\nModel loaded on {device}")
print(f"Mean/std shape: {mean.shape}")

Loading training config from: checkpoints/dfot/magnet_dd100/v0/checkpoints_500000/../config.yaml
Loading model from: checkpoints/dfot/magnet_dd100/v0/checkpoints_500000/model.safetensors
DFOTNetwork Cond
Loading model from checkpoints/vqvae/magnet_dd100/v0/checkpoints_100000/model.safetensors
Mode: Relative Canonical Cond

Model loaded on cuda
Mean/std shape: torch.Size([361])


## 3. Load Data

Load motion sequences from the HDF5 dataset.

In [5]:
# Map split name to DataType enum
split_map = {"test": DataType.TEST, "val": DataType.VAL, "train": DataType.TRAIN}
data_type = split_map[cfg.dataset_split]

print(f"Loading data from: {cfg.dataset_path}")
print(f"Split: {cfg.dataset_split}")
print(f"Sequences: {cfg.inference_data_list}")

test_data, test_person_num = load_from_hdf5(
    data_type=data_type,
    hdf5_path=cfg.dataset_path,
    mean_std_file_path=config.mean_std_path,
    file_list=cfg.inference_data_list,
    person_num=config.person_num,
    device=device,
    is_mask_additional_person=cfg.is_mask_additional_person,
)

# Optionally add person-flipped versions
if cfg.is_person_flip:
    additional_data = {}
    additional_person_num = {}
    for file_name, data in test_data.items():
        new_data_dict = {}
        for f in dataclasses.fields(data):
            new_data_dict[f.name] = getattr(data, f.name)[:, [1, 0]]
        additional_data[file_name + "_flip"] = TrainingData(**new_data_dict)
        additional_person_num[file_name + "_flip"] = test_person_num[file_name]
    test_data.update(additional_data)
    test_person_num.update(additional_person_num)

print(f"\nLoaded {len(test_data)} sequence(s):")
for name, data in test_data.items():
    n_frames = data.betas.shape[0]
    n_persons = test_person_num[name]
    print(f"  {name}: {n_frames} frames, {n_persons} person(s)")

Loading data from: data/duolandosmplx_dataset.hdf5
Split: test
Sequences: ['PasoDable_005_005', 'Qiaqiaqia_005_002']

Loaded 2 sequence(s):
  PasoDable_005_005: 1710 frames, 2 person(s)
  Qiaqiaqia_005_002: 2520 frames, 2 person(s)


## 4. Run Inference

Generate motion samples for each sequence using the DFOT model.

In [6]:
token_module = pose_vqvae.MultiPoseToken
pred_dict = {}
gt_motion_dict = {}

total = len(test_data)
for idx, (file_name, data) in enumerate(test_data.items(), 1):
    print(f"\n[{idx}/{total}] Generating: {file_name}")
    start_time = time.time()

    pred_motion, _ = model.sample_sequence(
        sampling_config=cfg.sampling_cfg,
        data=data,
    )

    elapsed = time.time() - start_time
    n_frames = pred_motion.betas.shape[1]
    print(f"  -> {n_frames} frames in {elapsed:.1f}s ({n_frames / elapsed:.1f} fps)")

    pred_dict[file_name] = pred_motion
    gt_motion_dict[file_name] = data

print(f"\nGeneration complete! {len(pred_dict)} sequence(s) processed.")


[1/2] Generating: PasoDable_005_005
context_seq_len: 32


Sampling:   2%|▏         | 9/500 [00:00<00:05, 87.09it/s]

cfg_scale_dict: {'clean': 1.0}


Sampling:  36%|███▌      | 178/500 [00:00<00:01, 267.32it/s]

[SmoothingGuidance] step=200/500 grad_norm=0.409784 weight=2.0
[FootSkatingGuidance] step=200/500 grad_norm=0.152689 weight=2.0


...

[SmoothingGuidance] step=300/500 grad_norm=0.471905 weight=2.0
[FootSkatingGuidance] step=300/500 grad_norm=0.196482 weight=2.0


Sampling:  81%|████████  | 405/500 [00:09<00:04, 22.21it/s]

[SmoothingGuidance] step=400/500 grad_norm=0.427001 weight=2.0
[FootSkatingGuidance] step=400/500 grad_norm=0.188670 weight=2.0


Sampling: 100%|██████████| 500/500 [00:11<00:00, 41.77it/s]

  -> 208 frames in 12.0s (17.3 fps)

Generation complete! 2 sequence(s) processed.


## 5. Post-process & Save

Denormalize predictions, convert root transforms from canonical to world space,
and save as `.npz` files (compatible with the Viser visualizer).

In [7]:
# Create output directory
date_str = time.strftime("%Y%m%d_%H%M%S")
checkpoint_dir_path = Path(cfg.checkpoint_dir)
experiment_name = checkpoint_dir_path.parents[1].name
prefix = cfg.result_dir_prefix or f"{cfg.epoch}_{cfg.dataset_split}"
out_dir = Path(f"{cfg.output_dir}/{experiment_name}/{prefix}_{date_str}")
os.makedirs(out_dir, exist_ok=True)

# Save inference config for reproducibility
OmegaConf.save(config=cfg, f=str(out_dir / "inf_cfg.yaml"))
print(f"Output directory: {out_dir}")

# Process and save each sequence
root_processor = RootTransformProcessor()

for file_name, raw_pred_motion in pred_dict.items():
    print(f"\nProcessing: {file_name}")

    # Denormalize predictions
    multi_pred_motion = token_module.denormalize(raw_pred_motion, mean, std)
    raw_gt_motion = gt_motion_dict[file_name]
    gt_timesteps = min(raw_gt_motion.betas.shape[0], raw_pred_motion.betas.shape[1])

    gt_motion = TrainingData.denormalize_unpacked(raw_gt_motion, mean, std)
    gt_motion = padding_training_data(
        data=gt_motion,
        target_len=multi_pred_motion.body_joint_rotations.shape[1],
    )

    # Convert root transforms to world space
    ctx_len = cfg.sampling_cfg.context_seq_len if cfg.sampling_cfg.context_seq_len > 0 else 4
    T_world_root = root_processor.convert_root_transform(
        pred_motion=multi_pred_motion,
        mode="temporal",
        context_seq_len=ctx_len,
        gt_motion=gt_motion,
    )

    # Prepend ground truth as first sample (index 0 = GT, indices 1..N = predictions)
    T_world_root = torch.cat(
        [SE3.from_9d(gt_motion.T_world_root).wxyz_xyz.unsqueeze(0), T_world_root], dim=0
    )
    body_joint_rotations = torch.cat(
        [SO3.from_6d(gt_motion.body_joint_rotations).wxyz.unsqueeze(0),
         SO3.from_6d(multi_pred_motion.body_joint_rotations).wxyz], dim=0
    )
    hand_joint_rotations = torch.zeros(
        body_joint_rotations.shape[:-2] + (30, 4),
        dtype=body_joint_rotations.dtype,
        device=body_joint_rotations.device,
    )
    betas = gt_motion.betas.unsqueeze(0).repeat(
        1 + multi_pred_motion.betas.shape[0], 1, 1, 1
    )

    # Slice to actual person count and save
    person_num = test_person_num[file_name]
    processed_data = {
        "betas":                betas[:, :, :person_num].numpy(force=True),
        "T_world_root":         T_world_root[:, :, :person_num].numpy(force=True),
        "body_joint_rotations": body_joint_rotations[:, :, :person_num].numpy(force=True),
        "hand_joint_rotations": hand_joint_rotations[:, :, :person_num].numpy(force=True),
        "context_timesteps":    cfg.sampling_cfg.context_seq_len,
        "gt_timesteps":         gt_timesteps,
        "timesteps":            betas.shape[1],
        "mode":                 cfg.sampling_cfg.sampling_task,
    }

    save_path = f"{out_dir}/{file_name}.npz"
    np.savez(save_path, **processed_data)

    n_samples = betas.shape[0] - 1  # exclude GT
    n_frames  = betas.shape[1]
    print(f"  Saved: {save_path}")
    print(f"  Shape: {n_samples} samples x {n_frames} frames x {person_num} person(s)")

print(f"\nAll results saved to: {out_dir}")

Output directory: outputs/dfot/magnet_dd100/500K_test_20260318_113833

Processing: PasoDable_005_005
mode temporal
  Saved: outputs/dfot/magnet_dd100/500K_test_20260318_113833/PasoDable_005_005.npz
  Shape: 10 samples x 208 frames x 2 person(s)

Processing: Qiaqiaqia_005_002
mode temporal
  Saved: outputs/dfot/magnet_dd100/500K_test_20260318_113833/Qiaqiaqia_005_002.npz
  Shape: 10 samples x 208 frames x 2 person(s)

All results saved to: outputs/dfot/magnet_dd100/500K_test_20260318_113833


## 6. Viser 3D Visualization

Launch the interactive **Viser** 3D viewer to inspect generated motions.

The viewer starts a local web server. Click the printed URL to open the interactive
3D scene in your browser. Features include:

- **Playback mode** — scrub through time with a slider, play/pause animation
- **Multitimestep mode** — show multiple frames simultaneously as a motion trail
- **GT vs Predicted** — toggle ground truth (gray) and predictions (colored)
- **Multiple samples** — view and compare different generated samples
- **File browser** — load different `.npz` files from the output directory
- **Hand pose** — switch between zero pose and fist pose for hands

### 6a. Configure the viewer

In [8]:
# ---------- Viser viewer settings ----------
VISER_PORT = 8084              # Port for the web viewer
VISER_DATA_DIR = str(out_dir)  # Directory with .npz files to visualize

# To visualize a different/older output directory, override VISER_DATA_DIR:
# VISER_DATA_DIR = "./outputs/dfot/magnet_dd100/500K_test_joint_YYYYMMDD_HHMMSS"

print(f"Data directory: {VISER_DATA_DIR}")
print(f"Port:           {VISER_PORT}")
print(f"\n.npz files found:")
for f in sorted(Path(VISER_DATA_DIR).glob("**/*.npz")):
    print(f"  {f.relative_to(VISER_DATA_DIR)}")

Data directory: outputs/dfot/magnet_dd100/500K_test_20260318_113833
Port:           8084

.npz files found:
  PasoDable_005_005.npz
  Qiaqiaqia_005_002.npz


### 6b. Launch the Viser server

Running this cell starts the Viser server. Open the printed URL in your browser.

**This cell blocks** — the viewer runs in a loop. To stop it:
- **Interrupt the kernel** (Kernel > Interrupt / stop button)
- Then continue with the cells below

In [ ]:
import viser
from libs.viz.visualizer import load_visualizer, visualize

data_root_dir = Path(VISER_DATA_DIR)
assert data_root_dir.exists(), f"Data directory not found: {data_root_dir}"

server = viser.ViserServer(port=VISER_PORT)
actual_port = server._port if hasattr(server, "_port") else VISER_PORT

# Request a public share URL (works from any browser, no SSH tunnel needed)
share_url = server.request_share_url()
print(f"\nViser server started!")
# print(f"  Local URL:  http://localhost:{actual_port}")
print(f"  Share URL:  {share_url}")
print(f"\nPress the stop/interrupt button to shut down the viewer.\n")

try:
    visualize(server, data_root_dir=data_root_dir)
except KeyboardInterrupt:
    print("\nViewer stopped.")

╭────── viser (listening *:8084) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8084   │
│   Websocket │ ws://localhost:8084     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://natural-subject.share.viser.studio


Viser server started!
  Share URL:  https://natural-subject.share.viser.studio

Press the stop/interrupt button to shut down the viewer.



(viser) Connection opened (0, 1 total), 63 persistent messages

task mode: joint
updating smpl data
updated smpl data
update visible sample idx list: [0, 1]
update visible sample idx list: [0, 1, 2]
update visible sample idx list: [1, 2]
update visible sample idx list: [2]

Viewer stopped.


(viser) Connection closed (0, 0 total)

(viser) Disconnected from share URL